# Spatial transcriptomics in Python: a first Visium analysis

**Audience:** Python users who are new to spatial transcriptomics.

**Prerequisites:** basic Python, pandas, and plotting. No single-cell experience is assumed.

**Biological question:** do gene-expression patterns and spatial relationships recover known anatomical regions in a mouse brain section?

By the end, you will be able to explain a Visium spot, inspect an `AnnData` object, evaluate spot-level QC, build a spatial-neighbor graph, interpret neighborhood enrichment, and use Moran's I to find spatially patterned genes.


## Mental model

A standard RNA-seq matrix tells us **what** genes are expressed. Spatial transcriptomics also records **where** each measurement came from. In this Visium dataset:

- each observation is a capture **spot**, not necessarily one cell;
- each variable is a gene;
- `adata.X` stores expression values;
- `adata.obs` and `adata.var` store spot and gene metadata;
- `adata.obsm['spatial']` stores each spot's x/y tissue coordinates.

The last item makes spatial questions possible.


## Outline

1. Load a public mouse-brain Visium dataset
2. Inspect expression and spatial structure
3. Calculate and visualize QC metrics
4. View annotations and marker genes on tissue
5. Build a spatial-neighbor graph
6. Test neighborhood enrichment
7. Rank spatially patterned genes with Moran's I
8. Interpret limitations and complete an exercise


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from spatial_demo import describe_spatial_adata

sc.settings.set_figure_params(dpi=90, facecolor="white", figsize=(6, 5))
sns.set_theme(style="whitegrid")
print(f"scanpy={sc.__version__}; squidpy={sq.__version__}")


## 1. Load the data

Squidpy distributes a preprocessed 10x Visium mouse-brain section used in its official tutorial. The first run downloads roughly 314 MB. Using preprocessed data lets this lesson focus on spatial concepts; a later lesson should start from raw counts and make filtering and normalization choices explicitly.


In [ ]:
adata = sq.datasets.visium_hne_adata()
summary = describe_spatial_adata(adata)
pd.Series(summary, name="value")


**Interpretation:** `n_spots` is the number of spatial measurements and `n_genes` is the number of measured features. There should be one `(x, y)` coordinate per spot. This dataset already contains expert-informed anatomical cluster labels; we use them to learn spatial statistics, not claim that we discovered them.


## 2. Spot-level quality control

Common QC summaries are total expression, detected genes, and mitochondrial fraction per spot. Extreme values can indicate damaged tissue, empty regions, technical artifacts, or real biology. Thresholds should be justified from distributions and tissue location—not copied blindly.

Because this teaching dataset is already preprocessed, these values are descriptive and we do **not** filter it here.


In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, inplace=True)
qc_columns = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
adata.obs[qc_columns].describe().round(2)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for column, ax in zip(qc_columns, axes):
    sns.histplot(adata.obs[column], bins=40, ax=ax)
    ax.set_title(column.replace("_", " "))
plt.tight_layout()


In [ ]:
sq.pl.spatial_scatter(adata, color=qc_columns, ncols=3, size=1.2)


**Questions to ask:** Are low-quality spots concentrated at tissue edges? Does a high-count region follow anatomy, suggesting biology rather than a technical artifact? Spatial QC plots preserve context that histograms lose.


## 3. Put expression back onto the tissue

A spatial cluster groups spots with similar molecular profiles, but a cluster is not automatically a cell type. In spot-based Visium data, each spot can mix several cells. Anatomical labels require evidence from marker genes, histology, and reference atlases.


In [ ]:
sq.pl.spatial_scatter(adata, color="cluster", size=1.25, figsize=(9, 7))


In [ ]:
candidate_genes = [g for g in ["Olfm1", "Plp1", "Itpka"] if g in adata.var_names]
sq.pl.spatial_scatter(adata, color=candidate_genes, ncols=3, size=1.3)


**Interpretation:** a convincing spatial marker forms a coherent tissue pattern rather than isolated high-expression spots. A pattern is evidence for spatial organization, not by itself proof of mechanism.


## 4. Build a spatial-neighbor graph

A graph represents each spot as a node and nearby spots as edges. On a Visium grid, Squidpy uses the known lattice arrangement. This lets us ask whether labels or expression are arranged non-randomly in space.


In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="grid")
connectivities = adata.obsp["spatial_connectivities"]
degrees = connectivities.sum(axis=1).A1
pd.Series(degrees, name="spatial_neighbors").describe().round(2)


Interior spots generally have more neighbors than edge spots. Always verify graph construction: grid adjacency, distance, and k-nearest neighbors encode different assumptions.


## 5. Neighborhood enrichment

Neighborhood enrichment compares observed label-to-label contacts with contacts expected after permuting labels. Positive z-scores mean two regions touch more often than expected; negative scores indicate avoidance. This is association, not evidence of cellular communication.


In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="cluster", n_perms=100, seed=42)
sq.pl.nhood_enrichment(adata, cluster_key="cluster", method="ward", figsize=(8, 8))


## 6. Spatially variable genes with Moran's I

Moran's I measures whether similar values occur near one another. Values above the random expectation indicate positive spatial autocorrelation. We test a subset of highly variable genes to keep the tutorial quick. Adjusted p-values matter because many genes are evaluated.


In [ ]:
if "highly_variable" in adata.var:
    tested_genes = adata.var_names[adata.var["highly_variable"]][:200].tolist()
else:
    tested_genes = adata.var_names[:200].tolist()

sq.gr.spatial_autocorr(
    adata, mode="moran", genes=tested_genes, n_perms=50, n_jobs=1, seed=42
)
moran_results = adata.uns["moranI"].copy()
moran_results.head(10)


In [ ]:
top_spatial_genes = moran_results.head(3).index.tolist()
sq.pl.spatial_scatter(adata, color=top_spatial_genes, ncols=3, size=1.3)


**Interpretation:** high Moran's I says expression is spatially structured. It does not say why. Anatomy, cell-type composition, technical gradients, or spatial regulation could all cause the pattern.


## Exercise: investigate one anatomical region

1. Choose a cluster from `adata.obs['cluster']`.
2. Report its number and percentage of spots.
3. Find the cluster with which it has the highest off-diagonal enrichment z-score.
4. In 2–3 sentences, explain what the result does and does not establish.

Try this yourself before revealing the scaffold below.


In [ ]:
# Exercise answer scaffold
chosen_cluster = "Hippocampus"
cluster_counts = adata.obs["cluster"].value_counts()
n_chosen = int(cluster_counts[chosen_cluster])
pct_chosen = 100 * n_chosen / adata.n_obs

enrichment = adata.uns["cluster_nhood_enrichment"]
cluster_names = adata.obs["cluster"].cat.categories
z_table = pd.DataFrame(enrichment["zscore"], index=cluster_names, columns=cluster_names)
best_neighbor = z_table.loc[chosen_cluster].drop(chosen_cluster).idxmax()

print(f"{chosen_cluster}: {n_chosen} spots ({pct_chosen:.1f}%)")
print(f"Strongest enriched neighbor: {best_neighbor}")
# Interpretation: These labels contact one another more often than expected
# under label permutation. This does not prove direct cell-cell signaling.


## Common pitfalls and next steps

- **Spot ≠ cell:** Visium spots can mix cell types, so avoid single-cell claims.
- **Cluster ≠ cell type:** annotate with markers, histology, and references.
- **Spatial association ≠ mechanism:** adjacency does not prove communication.
- **Preprocessed ≠ valid for every question:** inspect transformations and raw-count availability before differential-expression testing.
- **One section ≠ replication:** biological conclusions require multiple sections or subjects.

**Optional extension:** choose one region, identify marker genes with Scanpy, then compare them with a trusted mouse-brain atlas. The next lesson should repeat QC and preprocessing from raw Visium counts.


## Knowledge check

1. Why is a Visium spot not equivalent to a cell?
2. What does the spatial-neighbor graph add to expression data?
3. What does a high Moran's I mean—and not mean?
4. Why plot QC metrics on tissue coordinates?
5. What evidence is needed to call a cluster a cell type?
